# Model Parallelism — Splitting Models Across GPUs

When a model doesn't fit on a single GPU (or when you need more bandwidth/compute),
you split it across multiple GPUs. There are several ways to cut, each with different
tradeoffs.

This notebook covers:
1. Why we need parallelism — model sizes vs GPU memory
2. Tensor parallelism (TP) — split each layer across GPUs
3. Pipeline parallelism (PP) — assign different layers to different GPUs
4. Data parallelism (DP) — replicate the model, split the batch
5. Combining them — 3D parallelism in practice
6. Communication costs and interconnects
7. Expert parallelism — Mixture of Experts (MoE)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. Why We Need Parallelism

### The memory wall: models outgrow GPUs

| Model | Params | FP16 size | INT4 size | Fits on A100-80GB? |
|-------|--------|-----------|-----------|-------------------|
| LLaMA-3-8B | 8B | 16 GB | 4.5 GB | Yes (single GPU) |
| LLaMA-3-70B | 70B | 140 GB | 38 GB | No (FP16) / Barely (INT4) |
| LLaMA-3-405B | 405B | 810 GB | 220 GB | No (needs 3-10 GPUs) |
| GPT-4 class | ~1.8T | 3.6 TB | ~1 TB | No (needs 10-20+ GPUs) |

### Beyond memory: bandwidth parallelism

Even if a model fits on one GPU, you might want TP for **more aggregate bandwidth**:

```
1 × H100: 3.35 TB/s → 249 tok/s (7B FP16)
2 × H100: 6.70 TB/s → ~470 tok/s (same model, split across 2 GPUs)
4 × H100: 13.4 TB/s → ~880 tok/s (if communication doesn't bottleneck)
```

Parallelism gives you both **memory capacity** and **aggregate bandwidth**.

## 2. Tensor Parallelism (TP) — Split Each Layer

Tensor parallelism divides individual weight matrices across GPUs. Each GPU computes
a portion of the matmul, then they communicate to combine results.

### How it works for a linear layer

```
Weight matrix W: [2048, 6144]
Split across 4 GPUs by columns:

GPU 0: W[:, 0:1536]     → computes partial output [batch, 1536]
GPU 1: W[:, 1536:3072]  → computes partial output [batch, 1536]
GPU 2: W[:, 3072:4608]  → computes partial output [batch, 1536]
GPU 3: W[:, 4608:6144]  → computes partial output [batch, 1536]

Then: AllGather to combine → full output [batch, 6144]
```

### Column-parallel vs row-parallel

```
Column-parallel (split output dim):      Row-parallel (split input dim):

    X ─────┬──────┬──────┐                   X split
           │      │      │                ┌────┬────┬────┐
         [W₁]  [W₂]  [W₃]               [W₁] [W₂] [W₃]
           │      │      │                  │    │    │
         [Y₁]  [Y₂]  [Y₃]               [Y₁] [Y₂] [Y₃]
           │      │      │                  │    │    │
           └──AllGather──┘                  └──AllReduce──┘
                 │                                │
           Y (full)                          Y (full)
```

In Megatron-style TP (used by all major frameworks), the MLP uses:
- **Column-parallel** for the up-projection (split output → no communication needed between)
- **Row-parallel** for the down-projection (AllReduce to sum partial results)

Net effect: **one AllReduce per MLP block, one AllReduce per attention block**.

### Why TP helps decode latency

Each GPU only loads `1/TP` of the weights from its local HBM:
```
TP=1: load 14 GB from one GPU's HBM  → 14 GB / 3.35 TB/s = 4.2 ms
TP=4: load 3.5 GB from each GPU's HBM → 3.5 GB / 3.35 TB/s = 1.05 ms
                                         + AllReduce overhead (~0.1 ms)
                                         ≈ 1.15 ms total → 3.6x faster
```

Diminishing returns: communication overhead grows with TP degree.

In [ ]:
# Simulate TP scaling

def tp_decode_latency(model_size_gb, bandwidth_per_gpu_tb_s, tp_degree, 
                      allreduce_latency_us=50, num_allreduces_per_layer=2, num_layers=32):
    """Estimate decode latency with tensor parallelism."""
    # Each GPU loads 1/TP of the weights
    bytes_per_gpu = model_size_gb * 1e9 / tp_degree
    load_time_ms = bytes_per_gpu / (bandwidth_per_gpu_tb_s * 1e12) * 1000
    
    # Communication: AllReduce per attention + per MLP, per layer
    # AllReduce time depends on message size and interconnect
    comm_time_ms = num_allreduces_per_layer * num_layers * allreduce_latency_us / 1000
    
    total_ms = load_time_ms + comm_time_ms
    return total_ms, load_time_ms, comm_time_ms

tp_degrees = [1, 2, 4, 8]
model_gb = 14  # 7B in FP16
bw = 3.35  # H100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# With fast interconnect (NVLink: ~50us AllReduce)
latencies_nvlink = []
compute_parts = []
comm_parts = []
for tp in tp_degrees:
    total, compute, comm = tp_decode_latency(model_gb, bw, tp, allreduce_latency_us=50)
    latencies_nvlink.append(total)
    compute_parts.append(compute)
    comm_parts.append(comm)

# With slow interconnect (PCIe: ~200us AllReduce)
latencies_pcie = []
for tp in tp_degrees:
    total, _, _ = tp_decode_latency(model_gb, bw, tp, allreduce_latency_us=200)
    latencies_pcie.append(total)

ax1.bar(np.arange(len(tp_degrees)) - 0.15, compute_parts, 0.3, label='Weight load', color='steelblue')
ax1.bar(np.arange(len(tp_degrees)) + 0.15, comm_parts, 0.3, label='Communication', color='orange')
ax1.set_xticks(range(len(tp_degrees)))
ax1.set_xticklabels([f'TP={tp}' for tp in tp_degrees])
ax1.set_ylabel('Time (ms)')
ax1.set_title('Decode latency breakdown (7B FP16, H100 + NVLink)')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Speedup comparison
speedup_nvlink = [latencies_nvlink[0] / l for l in latencies_nvlink]
speedup_pcie = [latencies_pcie[0] / l for l in latencies_pcie]
ideal = tp_degrees

ax2.plot(tp_degrees, ideal, 'k--', label='Ideal (linear)', linewidth=1.5)
ax2.plot(tp_degrees, speedup_nvlink, 'g-o', label='NVLink (50us AllReduce)', linewidth=2)
ax2.plot(tp_degrees, speedup_pcie, 'r-s', label='PCIe (200us AllReduce)', linewidth=2)
ax2.set_xlabel('TP degree')
ax2.set_ylabel('Speedup vs TP=1')
ax2.set_title('TP scaling — interconnect matters')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"TP=4 with NVLink: {latencies_nvlink[2]:.2f} ms/token ({1000/latencies_nvlink[2]:.0f} tok/s)")
print(f"TP=4 with PCIe:   {latencies_pcie[2]:.2f} ms/token ({1000/latencies_pcie[2]:.0f} tok/s)")
print(f"\nNVLink gives {latencies_pcie[2]/latencies_nvlink[2]:.1f}x better scaling at TP=4.")
print(f"This is why DGX systems (8×H100 with NVLink) exist — interconnect is critical.")

## 3. Pipeline Parallelism (PP) — Stack Layers Across GPUs

Instead of splitting each layer, assign entire layers to different GPUs:

```
PP=4 for a 32-layer model:

GPU 0: Layers  0-7   (embedding + first 8 layers)
GPU 1: Layers  8-15
GPU 2: Layers 16-23
GPU 3: Layers 24-31  (+ lm_head)

Data flows: GPU 0 → GPU 1 → GPU 2 → GPU 3 (pipeline)
```

### The bubble problem

With a single request, only one GPU is active at a time:

```
Time →
GPU 0: [████]  idle   idle   idle
GPU 1:  idle  [████]  idle   idle
GPU 2:  idle   idle  [████]  idle
GPU 3:  idle   idle   idle  [████]

Utilisation: 25%!  ("pipeline bubble")
```

### Microbatching fills the pipeline

Split the batch into microbatches that flow through the pipeline:

```
Time →
GPU 0: [mb1] [mb2] [mb3] [mb4]  idle  idle  idle
GPU 1:  idle [mb1] [mb2] [mb3] [mb4]  idle  idle
GPU 2:  idle  idle [mb1] [mb2] [mb3] [mb4]  idle
GPU 3:  idle  idle  idle [mb1] [mb2] [mb3] [mb4]

Utilisation: much better! (bubble only at start/end)
```

### TP vs PP — when to use which

| | Tensor Parallelism | Pipeline Parallelism |
|---|---|---|
| Communication | AllReduce every layer (frequent, small) | Point-to-point between stages (infrequent, larger) |
| Latency impact | Reduces latency (parallel compute) | Doesn't reduce latency (sequential) |
| Throughput | Limited by communication overhead | Good throughput with microbatching |
| Needs | Fast interconnect (NVLink) | Any interconnect (even cross-node) |
| Best for | Latency-sensitive decode | Throughput / fitting very large models |

**Rule of thumb**: Use TP within a node (where NVLink connects GPUs), PP across nodes.

In [ ]:
# Visualise pipeline parallelism with and without microbatching

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6))

num_gpus = 4
stage_time = 1  # arbitrary time unit

# Without microbatching (single batch)
colors_single = ['#4CAF50']
for gpu in range(num_gpus):
    ax1.barh(gpu, stage_time, left=gpu * stage_time, color=colors_single[0], 
             edgecolor='black', linewidth=0.5)

ax1.set_yticks(range(num_gpus))
ax1.set_yticklabels([f'GPU {i}' for i in range(num_gpus)])
ax1.set_xlabel('Time')
ax1.set_title(f'Pipeline Parallelism: Single batch — {1/num_gpus*100:.0f}% GPU utilisation')
ax1.set_xlim(0, 8)
ax1.grid(True, alpha=0.2, axis='x')

# With microbatching (4 microbatches)
num_microbatches = 6
colors_mb = plt.cm.Set3(np.linspace(0, 1, num_microbatches))

for mb in range(num_microbatches):
    for gpu in range(num_gpus):
        start = mb + gpu
        ax2.barh(gpu, stage_time * 0.9, left=start * stage_time, color=colors_mb[mb],
                 edgecolor='black', linewidth=0.5)

ax2.set_yticks(range(num_gpus))
ax2.set_yticklabels([f'GPU {i}' for i in range(num_gpus)])
ax2.set_xlabel('Time')
total_slots = num_microbatches + num_gpus - 1
active_slots = num_microbatches * num_gpus
total_possible = total_slots * num_gpus
util = active_slots / total_possible * 100
ax2.set_title(f'Pipeline Parallelism: {num_microbatches} microbatches — {util:.0f}% GPU utilisation')
ax2.set_xlim(0, total_slots + 1)
ax2.grid(True, alpha=0.2, axis='x')

# Legend
patches = [mpatches.Patch(color=colors_mb[i], label=f'μbatch {i+1}') for i in range(num_microbatches)]
ax2.legend(handles=patches, loc='upper right', ncol=3, fontsize=9)

plt.tight_layout()
plt.show()

print(f"Pipeline bubble overhead: {(num_gpus - 1) / (num_microbatches + num_gpus - 1) * 100:.1f}%")
print(f"With more microbatches, the bubble becomes negligible.")
print(f"But for decode (batch=1, one token at a time), PP gives no latency benefit.")

## 4. Data Parallelism (DP) — Replicate Model, Split Batch

The simplest form: every GPU holds a full copy of the model, each processes
different requests.

```
DP=4:

GPU 0: [full model copy]  ← processes requests 1-8
GPU 1: [full model copy]  ← processes requests 9-16
GPU 2: [full model copy]  ← processes requests 17-24
GPU 3: [full model copy]  ← processes requests 25-32
```

### For inference:
- No communication needed (each replica is independent)
- Perfect linear throughput scaling
- But: each GPU must fit the entire model
- Doesn't help latency (each request still uses one GPU)

### When to use DP for serving
- Model fits on one GPU
- You need throughput, not lower latency
- Simple to implement (just run N copies)

## 5. Combining Them — 3D Parallelism

Large-scale deployments use all three simultaneously:

```
Example: 405B model on 64 GPUs (8 nodes × 8 GPUs)

TP=8 (within each node, connected by NVLink)
PP=2 (across 2 groups of nodes)
DP=4 (4 independent replicas)

Total GPUs = TP × PP × DP = 8 × 2 × 4 = 64

┌─────── Replica 1 ────────┐  ┌─────── Replica 2 ────────┐
│ Node 0 (TP=8, PP stage 0)│  │ Node 2 (TP=8, PP stage 0)│
│ [GPU0..GPU7] Layers 0-39 │  │ [GPU0..GPU7] Layers 0-39 │
│          ↓ (inter-node)  │  │          ↓               │
│ Node 1 (TP=8, PP stage 1)│  │ Node 3 (TP=8, PP stage 1)│
│ [GPU0..GPU7] Layers 40-79│  │ [GPU0..GPU7] Layers 40-79│
└──────────────────────────┘  └──────────────────────────┘
      ...+ Replica 3, 4
```

### How to choose the split

| Dimension | Communication pattern | Bandwidth need | Where to place |
|-----------|----------------------|----------------|---------------|
| TP | AllReduce every layer | Highest | Within node (NVLink: 900 GB/s) |
| PP | Point-to-point between stages | Medium | Across nodes (InfiniBand: 400 Gb/s) |
| DP | None (during inference) | Zero | Anywhere |

### Serving configurations in practice

| Model | GPUs | Typical config | Reasoning |
|-------|------|---------------|----------|
| 7-8B | 1 | TP=1, PP=1, DP=N | Fits on one GPU |
| 70B (FP16) | 4-8 | TP=4-8, PP=1 | Need bandwidth for decode |
| 70B (INT4) | 1-2 | TP=1-2, PP=1 | 38GB fits on 1-2 GPUs |
| 405B | 8-16 | TP=8, PP=2 | Needs capacity + bandwidth |
| GPT-4 class | 64+ | TP=8, PP=8+, DP=N | Massive model |

In [ ]:
# Throughput and latency under different parallelism strategies

def serving_metrics(model_gb, num_gpus, tp, pp, dp, 
                    gpu_bw_tb_s=3.35, nvlink_allreduce_us=50):
    """Estimate throughput and latency for a given parallelism config."""
    assert tp * pp * dp == num_gpus
    
    # Latency (single request): determined by TP and PP
    weight_per_tp_gpu = model_gb / (tp * pp)  # each GPU's share
    load_time_ms = weight_per_tp_gpu * 1e9 / (gpu_bw_tb_s * 1e12) * 1000
    
    # TP communication (per PP stage)
    num_allreduces = 2 * (32 // pp)  # 2 per layer (attn + MLP)
    tp_comm_ms = num_allreduces * nvlink_allreduce_us / 1000 if tp > 1 else 0
    
    # PP adds sequential stages (latency doesn't improve)
    latency_ms = (load_time_ms + tp_comm_ms) * pp  # sequential through PP stages
    
    # Throughput: DP replicas run independently
    tokens_per_sec_per_replica = 1000 / latency_ms
    total_throughput = tokens_per_sec_per_replica * dp
    
    return {
        "latency_ms": latency_ms,
        "throughput_tok_s": total_throughput,
        "tok_s_per_gpu": total_throughput / num_gpus,
    }

# Compare strategies for 70B FP16 on 8 GPUs
model_gb = 140  # 70B in FP16

strategies = [
    ("TP=8, PP=1, DP=1", 8, 1, 1),
    ("TP=4, PP=2, DP=1", 4, 2, 1),
    ("TP=4, PP=1, DP=2", 4, 1, 2),
    ("TP=2, PP=2, DP=2", 2, 2, 2),
    ("TP=2, PP=1, DP=4", 2, 1, 4),
]

print(f"70B model (FP16, 140 GB) on 8 × H100 GPUs:")
print(f"{'Strategy':<22} {'Latency (ms)':<14} {'Throughput':<14} {'Tok/s/GPU':<12} {'Best for'}")
print("─" * 75)

for name, tp, pp, dp in strategies:
    r = serving_metrics(model_gb, 8, tp, pp, dp)
    best_for = "Latency" if r['latency_ms'] == min(serving_metrics(model_gb, 8, *s[1:])["latency_ms"] for s in strategies) else \
              "Throughput" if r['throughput_tok_s'] == max(serving_metrics(model_gb, 8, *s[1:])["throughput_tok_s"] for s in strategies) else \
              "Balanced"
    print(f"{name:<22} {r['latency_ms']:<14.2f} {r['throughput_tok_s']:<14.0f} {r['tok_s_per_gpu']:<12.0f} {best_for}")

print(f"\nTP=8: lowest latency (best for real-time chat)")
print(f"TP=4,DP=2: best throughput (best for batch workloads)")
print(f"\nKey insight: TP helps latency, DP helps throughput. PP is for fitting large models.")

## 6. Communication Costs and Interconnects

Parallelism is only as good as the communication between GPUs.

### Interconnect hierarchy

```
┌───────────────────────────────────────────────────────────────────┐
│  Within a DGX H100 node (8 GPUs):                                │
│  NVLink 4.0: 900 GB/s bidirectional per GPU                      │
│  NVSwitch: full bisection bandwidth (any GPU to any GPU)         │
├───────────────────────────────────────────────────────────────────┤
│  Between nodes:                                                   │
│  InfiniBand NDR: 400 Gb/s (50 GB/s) per port                    │
│  8 ports per node → 400 GB/s aggregate                           │
├───────────────────────────────────────────────────────────────────┤
│  PCIe 5.0 (fallback):                                            │
│  64 GB/s per x16 slot                                            │
└───────────────────────────────────────────────────────────────────┘

NVLink is 18x faster than InfiniBand, 14x faster than PCIe.
This is why TP goes within a node and PP goes across nodes.
```

### AllReduce cost

For TP, each layer does an AllReduce of the activation tensor:
```
Message size per AllReduce = batch_size × seq_len × hidden_size × 2 bytes

At batch=1, 1 token, hidden=4096:
  Message = 1 × 1 × 4096 × 2 = 8 KB (tiny!)
  Latency-dominated: ~10-50 us regardless of size
  
At batch=32, seq=1, hidden=4096:
  Message = 32 × 1 × 4096 × 2 = 256 KB (still small)
  Still latency-dominated on NVLink
```

For decode, AllReduce messages are tiny — the overhead is mainly **latency** (launching
the communication), not bandwidth. This is why NVLink's low latency matters more
than its raw bandwidth for TP decode.

## 7. Expert Parallelism — Mixture of Experts (MoE)

MoE models (Mixtral, DeepSeek-V3, Qwen-MoE) have a unique parallelism opportunity:
most parameters are in "expert" MLP blocks, and only a subset is activated per token.

```
Dense model:        Every token → ALL parameters
MoE model (8e2a):   Every token → shared attention + 2 of 8 expert MLPs

Mixtral 8x7B:
  Total params: 47B
  Active params per token: ~13B (attention + 2 experts)
  
  Speed of a 13B model, quality of a 47B model!
```

### Expert parallelism (EP)

Assign different experts to different GPUs:

```
EP=4 for an 8-expert model:

GPU 0: Experts 0, 1  (shared attention replicated on all)
GPU 1: Experts 2, 3
GPU 2: Experts 4, 5
GPU 3: Experts 6, 7

Token routing:
  Token A → Experts 1, 5 → GPU 0 computes Expert 1, GPU 2 computes Expert 5
  Token B → Experts 3, 7 → GPU 1 computes Expert 3, GPU 3 computes Expert 7

Communication: All-to-All (send tokens to the GPU hosting their expert)
```

### EP vs TP for MoE

| | TP (split each expert) | EP (whole experts on different GPUs) |
|---|---|---|
| Communication | AllReduce per layer | All-to-All per layer |
| Load balance | Always balanced | Depends on routing |
| Memory | Each GPU stores 1/TP of all experts | Each GPU stores all of some experts |
| Best for | Small batch (latency) | Large batch (throughput) |

### The memory advantage of MoE

```
Dense 70B in FP16:   140 GB — needs TP=2-4 to serve
Mixtral 8x7B (47B):  94 GB total, but only 13B active
  → With EP=4: each GPU holds ~24 GB → fits easily
  → Decode speed: load only 13B active params = much faster
```

MoE gives you bigger model quality at smaller model serving cost.

In [ ]:
# Compare dense vs MoE serving economics

models = {
    "LLaMA-3-70B (Dense)": {
        "total_params_b": 70, "active_params_b": 70,
        "min_gpus_fp16": 4, "quality_score": 82,
    },
    "Mixtral 8x7B (MoE)": {
        "total_params_b": 47, "active_params_b": 13,
        "min_gpus_fp16": 2, "quality_score": 78,
    },
    "DeepSeek-V3 (MoE)": {
        "total_params_b": 671, "active_params_b": 37,
        "min_gpus_fp16": 16, "quality_score": 90,
    },
    "LLaMA-3-8B (Dense)": {
        "total_params_b": 8, "active_params_b": 8,
        "min_gpus_fp16": 1, "quality_score": 68,
    },
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

names = list(models.keys())
total = [m["total_params_b"] for m in models.values()]
active = [m["active_params_b"] for m in models.values()]

x = np.arange(len(names))
width = 0.35

ax1.bar(x - width/2, total, width, label='Total params (memory cost)', color='red', alpha=0.7)
ax1.bar(x + width/2, active, width, label='Active params (compute/bandwidth cost)', color='green', alpha=0.7)
ax1.set_xticks(x)
ax1.set_xticklabels([n.split(' (')[0] for n in names], rotation=15)
ax1.set_ylabel('Parameters (billions)')
ax1.set_title('Total vs active parameters')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.set_yscale('log')

# Cost efficiency: quality per active param
quality = [m["quality_score"] for m in models.values()]
efficiency = [q / a for q, a in zip(quality, active)]

ax2.bar(x, efficiency, color=['steelblue', 'green', 'darkgreen', 'lightblue'], alpha=0.7)
ax2.set_xticks(x)
ax2.set_xticklabels([n.split(' (')[0] for n in names], rotation=15)
ax2.set_ylabel('Quality / active params (higher = more efficient)')
ax2.set_title('Serving efficiency: quality per bandwidth dollar')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"DeepSeek-V3: 671B total params but only 37B active per token.")
print(f"  → Decode speed similar to a 37B dense model")
print(f"  → Quality of a much larger model")
print(f"  → Memory: needs to store 671B × 2 bytes = 1.3 TB (needs many GPUs for capacity)")
print(f"  → But bandwidth: only streams 37B × 2 bytes = 74 GB per token")

## Summary

| Strategy | What it splits | Helps with | Communication | Where |
|----------|---------------|------------|---------------|-------|
| **TP** | Each layer (columns/rows) | Latency + bandwidth | AllReduce every layer | Within node (NVLink) |
| **PP** | Layers across GPUs | Memory capacity | Point-to-point between stages | Across nodes |
| **DP** | Batch across replicas | Throughput | None (inference) | Anywhere |
| **EP** | Experts across GPUs | MoE memory + throughput | All-to-All per layer | Within/across nodes |

### Decision framework

```
Does the model fit on one GPU?
  YES → DP for throughput, done.
  NO  → How many GPUs to fit it?
         ≤8 (one node) → TP=N, PP=1
         >8 (multi-node) → TP=8 within node, PP across nodes
         
Need lower latency beyond TP scaling?
  → Quantise (INT4/INT8) to reduce model size, then use fewer GPUs
  → Or use speculative decoding to amortise the cost

MoE model?
  → EP for throughput at large batch, TP for latency at small batch
```